### 4. Building height calculation.

These notebook takes the output from notebook [No.2](http://localhost:8888/notebooks/notebooks/2_filter_and_extract_buildings_from_VIDA_S2_Partitions.ipynb), where some basic parameters where computed and the geotif files that we obtained from the [previous notebook](http://localhost:8888/notebooks/notebooks/3_download_google_25D.ipynb), to compute the height per building. The main function 'calculate_heights_in_tiff' computes building height statistics by cross-referencing building footprints with a corresponding elevation raster. It processes the raster in predefined tiles to handle potentially large files, calculates height metrics for each building within a tile, and returns a single, enriched DataFrame with the new height information.

**important:** The google building height layer has **three bands**: building_fractional_count, building_height, and building_presence. Thus, this notebook was designed to process the second band of the raster.

In [1]:
# Import necessary libraries
import io
from PIL import Image
from botocore.client import Config
import numpy as np
import configparser
import os
import sys
import pandas as pd
import geopandas as gpd
import random
import time
import base64
import shutil
import threading
from collections import Counter
from tqdm import tqdm
from datetime import datetime
import os
import rasterio
from rasterio.windows import Window
import gc
from matplotlib.path import Path
import matplotlib.pyplot as plt
import shapely
from rasterio.plot import show
import rioxarray
from tqdm import tqdm
import traceback
import json
from datetime import datetime

module_path = os.path.abspath(os.path.join('..', 'scripts'))
if module_path not in sys.path:
    sys.path.append(module_path)

from utils import *

import tkinter as tk
from tkinter import filedialog, messagebox
from datetime import datetime
import math

In [2]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-09-30 16:21:47'

In [3]:
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('Open Buildings Insights', 'Open the parquet file')
parquet_path = filedialog.askopenfilename()
messagebox.showinfo('Open Buildings Insights', 'Browse to the folder where you stored the Geotif files')
path_to_tif_folder= filedialog.askdirectory()
messagebox.showinfo('Open Buildings Insights', 'Browse to the folder where you will stored the output of this notebook')
path_to_parquet_output= filedialog.askdirectory()
main_df = pd.read_parquet(parquet_path)
main_df_len = len(main_df)

The next cell gives name to the region of study, it should matches the previous one.

In [4]:
region="Bhiwandi"

All of the coming functions arre embebed into the main one Calculating_heights_in_tiff:
* **reproject_tif_CRS:** It reprojects a GeoTIFF file to the standard WGS 84 latitude/longitude coordinate system and overwrites the original file with the result.It is memory intensive, if the infrastructure where is getting run, can't handle large datasets, it is necessary to break the tiffs into smaller rasters using the function: *tiling*
* **tiling**: It breaks down the second band of the input raster into a grid of n (num_tiels), giving as output n*n tiles
* **generate_imagin_coords**: This function divides a large raster into a grid of smaller, overlapping tiles and returns a list of pixel coordinates that define the boundaries for each of these tiles.
* **fetch_buildings_in_bbox**: It filters a pandas DataFrame to find all rows (buildings) that fall within a specified rectangular geographic area (a bounding box).
* **get_min_max_values_row_col**: it calculates the bounding box of a list of pixel coordinates by finding the minimum and maximum row and column values.
* **categorize_height**: It snaps the height dataset to the nearest logical:
    * If the height is NaN (not a number), it defaults to 4.5 meters.
    * Any height up to 4.5m is categorized as a single-story building and standardized to 4.5m.
    * Any height between 4.5m and 7.5m is categorized as a two-story building and standardized to 7.5m.
    * For any height above 7.5m, it rounds the height up to the next 3-meter increment (e.g., 8.5m becomes 10.5m, 11m becomes 13.5m, etc.)
* **calculate_floors_gfa** This function takes the clean, standardized height from the previous function, along with the building's footprint area, and calculates the final number of floors and the total GFA.

In [5]:
def reproject_tif_CRS(filename: str):
    with rioxarray.open_rasterio(filename) as rds:
        rds_4326 = rds.rio.reproject("EPSG:4326")
    rds_4326.rio.to_raster(filename, compress="DEFLATE")

def tiling(input_raster_path, output_dir, num_tiles=2):
    """
    Breaks a large multi-band raster file into a square grid of smaller,
    single-band tiles, using only the data from the second band.
    """
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(input_raster_path) as src:
        # Check if the raster has at least two bands
        if src.count < 2:
            print("Error: Input raster has fewer than two bands.")
            return

        width = src.width
        height = src.height
        
        tile_width = math.ceil(width / num_tiles)
        tile_height = math.ceil(height / num_tiles)

        output_filename_template = os.path.basename(input_raster_path).replace('.tif', '_band2_tile_{}-{}.tif')

        for col_off in range(0, width, tile_width):
            for row_off in range(0, height, tile_height):
                
                actual_width = min(tile_width, width - col_off)
                actual_height = min(tile_height, height - row_off)
                
                window = Window(col_off, row_off, actual_width, actual_height)
                
                transform = src.window_transform(window)
                
              
                data = src.read(2, window=window) ##Reading the second band from the input raster

                if src.nodata is not None and np.all(data == src.nodata):
                    continue

                profile = src.profile.copy()
                
                # --- CHANGE 2: Update metadata for a SINGLE-BAND output ---
                profile.update({
                    'count': 1, # The output raster will have only one band
                    'height': actual_height,
                    'width': actual_width,
                    'transform': transform
                })
                
                output_path = os.path.join(output_dir, output_filename_template.format(row_off, col_off))

                with rasterio.open(output_path, 'w', **profile) as dst:
                    # --- CHANGE 3: Write the 2D data array to the first band of the new file ---
                    dst.write(data, 1)
                    
    print(f"Tiling of band 2 for raster {os.path.basename(input_raster_path)} complete.")

def generate_image_coords(heights_tiff_name:str, div_arg:int = 1) -> list:
    
    dat = rasterio.open(os.path.join(path_to_tif_folder, heights_tiff_name))
    profile = dat.profile.copy()
    profile.update(compress='lzw')
    # print(profile)
    
    #divide tiff to tiles
    tiff_width = profile['width']
    tiff_height = profile['height']
    
    tile_width = int(tiff_width / div_arg)
    tile_height = int(tiff_height / div_arg)
    
    print(f'tile_width: {tile_width}, tile_height: {tile_height}')
    # define overlap between tiles
    overlap = 1000
    
    columns_amount = int(tiff_width / tile_width) if tiff_width % tile_width == 0 else int(tiff_width / tile_width) + 1
    rows_amount = int(tiff_height / tile_height) if tiff_height % tile_height == 0 else int(tiff_height / tile_height) + 1
    print(f'TIFf image wiil be divided to {rows_amount} rows and {columns_amount} cols')
    
    images_coords = []
    
    for col_idx in range(1, columns_amount + 1):
        
        row_start = max(tile_width * (col_idx - 1) - overlap, 0)
        
        if col_idx != columns_amount:
            
            row_limits = [row_start, tile_width * col_idx]
        elif col_idx == columns_amount:
            row_limits = [row_start, tiff_width]
        
        for row_idx in range(1, rows_amount + 1):
            
            col_start = max(tile_height * (row_idx - 1) - overlap, 0)
            
            if row_idx != columns_amount:
                col_limits = [col_start, tile_height * row_idx]
            elif row_idx == columns_amount:
                col_limits = [col_start, tiff_height]
                
            coords = [col_limits, row_limits]
            images_coords.append(coords)

    return images_coords

fetch_builings_in_bbox =\
lambda df, lon_min, lon_max, lat_min, lat_max:\
df[(df.longitude >= lon_min) & (df.longitude <= lon_max) & (df.latitude >= lat_min) & (df.latitude <= lat_max)]

def get_min_max_values_row_col(pixel_coordinates: list):
    
    return {
        'rowminmax': 
            [
                min([i[0] for i in pixel_coordinates]), 
                max([i[0] for i in pixel_coordinates]), 
            ], 
        'colminmax': 
            [
                min([i[1] for i in pixel_coordinates]), 
                max([i[1] for i in pixel_coordinates]), 
            ]
            }

def calculate_floors_gfa(height_categorized, area_in_meters):

    if np.isnan(height_categorized):
        height_categorized = 1
        
    def get_floor(height):
        if (height >= 0) and (height <= 4.5):
            return 1
        elif (height > 4.5) and (height <= 7.5):
            return 2
        elif (height > 7.5):
            return int(((height - 1.5005) // 3 * 3 + 3) / 3)

    floors = get_floor(height_categorized)
    gfa = round(area_in_meters * floors, 5)

    return floors, gfa


def categorize_height(height):
    if np.isnan(height):
        return 4.5
    height = round(height, 5)
    if (height >= 0) & (height <= 4.5):
        return 4.5
    elif (height > 4.5) & (height <= 7.5):
        return 7.5
    else:
        return (int(int(height - 4.5) // 3) * 3) + 7.5

The *calculate_heights_in_tiff* function is a geospatial processing tool designed to enrich a DataFrame of building footprints (main_df) with height information derived from a GeoTIFF file. To manage potentially large files, it first calls a helper function, generate_image_coords, which creates a plan to process the raster in smaller, overlapping chunks or "tiles." The main function then iterates through each of these tiles, first finding all the buildings from the master DataFrame that fall within the tile's geographic area. It then reads only that small corresponding chunk of the raster's pixel data into memory. For each building within the chunk, it creates a precise pixel mask of its footprint, uses the mask to extract the underlying elevation values, and calculates statistics such as the median and mean height. These values are then used by other helper functions to estimate and add new columns to the DataFrame. The final output is a single DataFrame which contains the new attributes: number of floors, gross floor area and height.

In [6]:
def calculate_heights_in_tiff(path_to_tif_folder,main_df, heights_tiff_name):

    t1 = time.time()
    
    tiff_path = os.path.join(path_to_tif_folder, heights_tiff_name)
    
    #print(tiff_path)

    
    if tiff_path != None:
        #print('reproject_tif_CRS')
        reproject_tif_CRS(tiff_path)

    images_coords = generate_image_coords(heights_tiff_name)
    dfs = []
    
    
    # loop through tiles coords
    for idx, coords in enumerate(images_coords):


        with rasterio.open(tiff_path) as src:
            profile = src.profile.copy()
            tiff_width = profile['width']
            tiff_height = profile['height']
            # read tiff metadata by coords in order to prepare filtered dataframe
            print(coords)
            
            col_off = coords[1][0]
            row_off = coords[0][0]
            
            width = coords[1][1] - coords[1][0]
            height = coords[0][1] - coords[0][0]
            
            
            lon_upper_left, lat_upper_left = src.xy(coords[0][0], coords[1][0])
            lon_down_right, lat_down_right = src.xy(coords[0][1], coords[1][1])
    
            lons_sorted = sorted([lon_upper_left, lon_down_right])
            lats_sorted = sorted([lat_upper_left, lat_down_right])
            
            lon_min = lons_sorted[0]
            lon_max = lons_sorted[1]
    
            lat_min = lats_sorted[0]
            lat_max = lats_sorted[1]
            
            areas_covered_by_tifs = create_bounds_dict(path_to_tifs=path_to_tif_folder)
    
            # set up default height (3m - 1 floor) for buildings under the threshold (20 square meters)
            # upd_default_height_in_bbox(lon_min, lon_max, lat_min, lat_max)
    
            # fetch all buildings larger than the threshold (20 square meters) to estimate its height
            df = fetch_builings_in_bbox(main_df, lon_min, lon_max, lat_min, lat_max).copy()
            df['geometry'] = df['geometry'].apply(shapely.from_wkb)
            init_len = len(df)
            # coordinates of current tile
            col_off = max(coords[1][0] - 1, 0)
            row_off = max(coords[0][0] - 1, 0)
            width = min(coords[1][1] - coords[1][0] + 1, tiff_width)
            height = min(coords[0][1] - coords[0][0] + 1, tiff_height)
            
            print('offsets', col_off, row_off)
            
            # read tile grayscale layer
            tiff_data = src.read(1, window=Window(col_off, row_off, width, height))
            
            tiff_data[tiff_data == -99.0] = np.nan
            
            print(f"Images revealed: {len(df)}")
            
            # loop through building centroids inside tile

            df.index = [i for i in range(len(df))]
            
            for index, row, in tqdm(df.iterrows(), total=len(df), desc='Height calculation'):
                try:
    
                    # lat lon to pixel transformations
                    pixel_coordinates = get_pixel_coordinates(row.geometry, areas_covered_by_tifs, src)
                    polygon_coordinates = [[pixel_coords[0] - row_off, pixel_coords[1] - col_off] for pixel_coords in pixel_coordinates]
                    
                    margin = 0
                    
                    rowcolminmax = get_min_max_values_row_col(polygon_coordinates)
                    
                    img_width = rowcolminmax['rowminmax'][1] - rowcolminmax['rowminmax'][0] 
                    img_height = rowcolminmax['colminmax'][1] - rowcolminmax['colminmax'][0]
                    
                    row_start = rowcolminmax['rowminmax'][0]
                    row_end = rowcolminmax['rowminmax'][1]
                    col_start = rowcolminmax['colminmax'][0]
                    col_end = rowcolminmax['colminmax'][1]

                    
                    # img_array_pre = np.array(tiff_data[row_start : row_end, col_start : col_end])
                    
    
                    # polygon_coordinates = offset_polygon_coords(polygon_coordinates)
                    # rowcolminmax = get_min_max_values_of_row_col(polygon_coordinates)
                    
                    # img_width = rowcolminmax['rowminmax'][1] - rowcolminmax['rowminmax'][0] 
                    # img_height = rowcolminmax['colminmax'][1] - rowcolminmax['colminmax'][0]
                    # row_start = rowcolminmax['rowminmax'][0]
                    # row_end = rowcolminmax['rowminmax'][1]
                    # col_start = rowcolminmax['colminmax'][0]
                    # col_end = rowcolminmax['colminmax'][1]

                    # cut building image from tile
                    img_array_pre = np.array(tiff_data[row_start : row_end, col_start : col_end])

                    if img_array_pre.shape != (img_width, img_height):
                        print('building polygon out of tiff')
                        np_nan_matrix = np.empty((img_width, img_height))
                        np_nan_matrix.fill(np.nan)
                        np_nan_matrix[:img_array_pre.shape[0], :img_array_pre.shape[1]] = img_array_pre
                        img_array_pre = np_nan_matrix
                    
                    # extract building by polygon coords
                    absolule_polygon_coordinates = [[pixel_coords[0] - row_start, pixel_coords[1] - col_start] for pixel_coords in polygon_coordinates]
                    poly_path=Path(absolule_polygon_coordinates)
                    x, y = np.mgrid[:img_height, :img_width]
                    coors = np.hstack((x.reshape(-1, 1), y.reshape(-1,1)))
                    mask = poly_path.contains_points(coors).reshape(img_height, img_width).T
                    
                    # create zeros mask
                    img_masked=np.zeros((img_width, img_height),dtype=img_array_pre.dtype)
    
                    # put image on zeros mask
                    img_masked[mask]=img_array_pre[mask]
    
                    # extract image as list of non zero values
                    
                    img_masked_list = list(filter(lambda num: num != 0, img_masked.flatten(order='C')))

                    if len(img_masked_list) > 0:
                        img_masked_list = [i for i in img_masked_list if not np.isnan(i)]
                        
                        if len(img_masked_list) == 0:
                            img_masked_list = np.array([0])
                    else:
                        img_masked_list = np.array([0])
                        
                    if len(img_masked_list) > 0:    
                        nanmedian_height = np.nanmedian(img_masked_list)
                        
                        df.at[index, 'height_mean'] = np.nanmean(img_masked_list)
                        df.at[index, 'height_median'] = nanmedian_height
                        df.at[index, 'height_max'] = np.max(img_masked_list)
                        df.at[index, 'height'] = categorize_height(nanmedian_height)
    
                        floors, gfa = calculate_floors_gfa(nanmedian_height, row.area_in_meters)
                        df.at[index, 'floors'] = floors
                        df.at[index, 'gfa_in_meters'] = gfa
                        
                except Exception as e:
                    # pass
                    print(f'Height calculation error {e}')
                    traceback.print_exc()
                
            dfs.append(df)
            
    try:
        result_df = pd.concat(dfs)

        result_len = len(result_df)
        print(f'Init len {init_len} -> result len {result_len} | diff: {init_len - result_len}')

        print('Remove', tiff_path)
        os.remove(tiff_path)
        return result_df

    except Exception as e:
        print(f'Concat or upload error occurred: {e}')


The next step is necesary to break down the tiles into smaller chunks.

In [7]:
tiffs_temporal = "tiffs_temporal"
os.makedirs(tiffs_temporal, exist_ok=True)
for filename in os.listdir(path_to_tif_folder):
    file_path = os.path.join(path_to_tif_folder, filename)
    if filename.lower().endswith(".tif"):
        tiling(file_path, tiffs_temporal)

root_path = os.getcwd()
path_to_tif_folder = os.path.join(root_path, tiffs_temporal)
country_tile_filenames = [f for f in os.listdir(path_to_tif_folder) if os.path.isfile(os.path.join(path_to_tif_folder, f))]

Tiling of band 2 for raster Bhiwandi_tile_0mLhQWmhXjQ.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_17R0NGdzmL4.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_4VNHnxne5qA.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_9saav0m0P7I.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_9ZUNu889D5g.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_GA7hNOyJRak.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_LoT1jPAFGko.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_uhCUB-kvkJ0.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_XEdgqNGqgYw.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_y0RJ1TudVeQ.tif complete.
Tiling of band 2 for raster Bhiwandi_tile_ZJUo2y3Nj6s.tif complete.
Tiling of band 2 for raster Bhiwandi_tile__tLbQHqJ_cg.tif complete.


The next part of the code iterates throug each tiff file and computes the calculate_heights_in_tiff function. The results are stored in parquet files with the prefix "lost_part_".

In [8]:
total_count = 0
for tidx, tiff_name in enumerate(country_tile_filenames):
# -1 to start from beginning
    if tidx > -1:
        tile_progress = f'Processing {tiff_name}  {tidx+1} of {len(country_tile_filenames)}'
        #print(tile_progress)
    
        heights_df = calculate_heights_in_tiff(path_to_tif_folder, main_df, tiff_name) #updated function including folder
        total_count += len(heights_df)
        
        if len(heights_df) > 0:
            try:
                filename = tiff_name.replace('.tif', '.parquet')
                filename = f'lost_part_{filename}'
                heights_df = gpd.GeoDataFrame(heights_df, geometry=heights_df.geometry)
                heights_df.to_parquet(os.path.join(root_path, filename)) #change to save parquet files in the root_directory
            except Exception as e:
                parquet_upload_status = f'Parquet: {filename} not processed error: {e}'
    
            state = dict(
                calculated_count = total_count,
                progress = f'{100*round(total_count/main_df_len, 6)}% | {total_count} of {main_df_len}',
                #parquet_upload_status = parquet_upload_status,
                tile = tile_progress
            )

            print()
os.rmdir(path_to_tif_folder)

tile_width: 12953, tile_height: 12281
TIFf image wiil be divided to 1 rows and 1 cols
[[0, 12281], [0, 12953]]
offsets 0 0
Images revealed: 0


Height calculation: 0it [00:00, ?it/s]


Init len 0 -> result len 0 | diff: 0
Remove C:\Users\renec\OneDrive\Documents\SEForALL\Local\OBI Github\data-preprocessing-local\notebooks\tiffs_temporal\Bhiwandi_tile_0mLhQWmhXjQ_band2_tile_0-0.tif
tile_width: 12953, tile_height: 12281
TIFf image wiil be divided to 1 rows and 1 cols
[[0, 12281], [0, 12953]]
offsets 0 0
Images revealed: 0


Height calculation: 0it [00:00, ?it/s]

Init len 0 -> result len 0 | diff: 0
Remove C:\Users\renec\OneDrive\Documents\SEForALL\Local\OBI Github\data-preprocessing-local\notebooks\tiffs_temporal\Bhiwandi_tile_0mLhQWmhXjQ_band2_tile_0-12953.tif


...

building polygon out of tiff


Height calculation: 100%|████████████████████████████████████████████████████████████████| 2918/2918 [00:25<00:00, 113.70it/s]


Init len 2918 -> result len 2918 | diff: 0
Remove C:\Users\renec\OneDrive\Documents\SEForALL\Local\OBI Github\data-preprocessing-local\notebooks\tiffs_temporal\Bhiwandi_tile__tLbQHqJ_cg_band2_tile_12500-0.tif

tile_width: 12948, tile_height: 12282
TIFf image wiil be divided to 1 rows and 1 cols
[[0, 12282], [0, 12948]]
offsets 0 0
Images revealed: 306


Height calculation: 100%|██████████████████████████████████████████████████████████████████| 306/306 [00:02<00:00, 104.55it/s]


Init len 306 -> result len 306 | diff: 0
Remove C:\Users\renec\OneDrive\Documents\SEForALL\Local\OBI Github\data-preprocessing-local\notebooks\tiffs_temporal\Bhiwandi_tile__tLbQHqJ_cg_band2_tile_12500-12500.tif



Each iteration will create some parquet files that were name as "lost_part_", this last sted combines them into a single dataset and gets rid of duplicates.

In [9]:
all_parquet_files = [f for f in os.listdir(root_path) if os.path.isfile(os.path.join(root_path, f))]
parquets_to_merge = [i for i in all_parquet_files if ('lost_part_'+region in i) and ('.parquet' in i)]
df_list = [pd.read_parquet(os.path.join(root_path,file)) for file in parquets_to_merge]
merged_df = pd.concat(df_list, ignore_index=True)

# Drop exact duplicates
merged_df.drop_duplicates(inplace=True)

# Save the merged dataset to a new Parquet file
merged_df.to_parquet(os.path.join(path_to_parquet_output,region + "_height_NEW.parquet"), index=False)

print(f"Merged {len(parquets_to_merge)} files into '{region}_height.parquet' with {len(merged_df)} unique rows.")

#delete single parquet files
for file in parquets_to_merge:
    os.remove(os.path.join(root_path, file))

Merged 32 files into 'Bhiwandi_height.parquet' with 221536 unique rows.


In [10]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-09-30 17:44:01'